
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# 2L - Grouping and Aggregating E-Commerce Data

In this lab, you'll practice working with grouping and aggregation in Spark using a dataset of e-commerce transactions. You'll perform various analyses to uncover patterns and insights in customer purchasing behavior.

### Objectives
- Use `groupBy` operations to summarize data
- Implement multiple aggregations
- Apply different ordering techniques
- (Bonus) Use window functions for advanced analytics

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.

1. Wait a few minutes for the cluster to start.

1. Once the cluster is running, complete the steps above to select your cluster.

## A. Initial Setup

Load the retail transactions data and examine its structure.

In [0]:
from pyspark.sql.functions import *

## Read the e-commerce transactions data
transactions_df = spark.read.table("samples.bakehouse.sales_transactions")

## display a sample of the data
transactions_df.printSchema()

display(transactions_df.limit(10))

root
 |-- transactionID: long (nullable = true)
 |-- customerID: long (nullable = true)
 |-- franchiseID: long (nullable = true)
 |-- dateTime: timestamp (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unitPrice: long (nullable = true)
 |-- totalPrice: long (nullable = true)
 |-- paymentMethod: string (nullable = true)
 |-- cardNumber: long (nullable = true)



transactionID,customerID,franchiseID,dateTime,product,quantity,unitPrice,totalPrice,paymentMethod,cardNumber
2002961,1000253,3000047,2024-05-14T12:17:01.495952Z,Golden Gate Ginger,8,3,24,amex,378154478982993
2003007,1000226,3000047,2024-05-10T23:10:10.239954Z,Austin Almond Biscotti,36,3,108,mastercard,2244626981238094
2003017,1000108,3000047,2024-05-16T16:34:10.61372Z,Austin Almond Biscotti,40,3,120,mastercard,2490570234487424
2003068,1000173,3000047,2024-05-02T04:31:51.612094Z,Pearly Pies,28,3,84,amex,343808569426192
2003103,1000075,3000047,2024-05-04T23:44:26.902224Z,Pearly Pies,28,3,84,visa,4377080942201798
2003147,1000295,3000047,2024-05-15T16:17:06.25945Z,Austin Almond Biscotti,32,3,96,amex,371093774812677
2003196,1000237,3000047,2024-05-07T11:13:22.469231Z,Tokyo Tidbits,40,3,120,mastercard,5538807345848392
2003329,1000272,3000047,2024-05-06T03:32:16.017968Z,Outback Oatmeal,28,3,84,visa,4872480716880043
2001264,1000209,3000047,2024-05-16T17:32:28.547589Z,Pearly Pies,28,3,84,mastercard,5287105980593305
2001287,1000120,3000047,2024-05-15T08:41:28.406738Z,Austin Almond Biscotti,40,3,120,amex,376211012259783


## B. Basic Grouping Operations

Let's start with simple grouping operations to understand product sales patterns.

In [0]:
# 1. Group the data by products and count the number of sales
# 2. Order the results by the most popular products

In [0]:
## Count transactions by product
product_counts = transactions_df \
    .groupBy("product") \
    .count() \
    .orderBy(desc("count"))

display(product_counts)

product,count
Golden Gate Ginger,586
Tokyo Tidbits,583
Outback Oatmeal,561
Pearly Pies,550
Austin Almond Biscotti,530
Orchard Oasis,523


## C. Multiple Aggregations

Now let's perform multiple aggregations to get deeper insights.

In [0]:
# 1. Analyze sales by payment method
# 2. Calculate the total revenue, average transaction value, and count of transactions for each payment method
# 3. Order by total revenue (highest first)

In [0]:
## Analyze sales by payment method
payment_analysis = transactions_df \
    .groupBy("paymentMethod") \
    .agg(
        round(sum(col("totalPrice")), 2).alias("total_revenue"),
        round(avg(col("totalPrice")), 2).alias("avg_transaction_value"),
        count("*").alias("transaction_count")
    ) \
    .orderBy(desc("total_revenue"))

display(payment_analysis)

paymentMethod,total_revenue,avg_transaction_value,transaction_count
amex,22434,20.28,1106
mastercard,22263,19.46,1144
visa,21774,20.11,1083


## Bonus Challenge: Window Functions

If you have time, try using window functions for advanced analytics.

In [0]:

## First, calculate total revenue by product and 
product_revenue_df = transactions_df \
    .groupBy("product") \
    .agg(
        round(sum(col("totalPrice")), 2).alias("total_revenue")
    )

## Using window functions to add rankings
## Ranking products by total revenue

from pyspark.sql.window import Window

## Create window spec for ranking categories
window_by_revenue = Window.orderBy(desc("total_revenue"))

## Add rankings
ranked_products_df = product_revenue_df \
    .withColumn("revenue_rank", rank().over(window_by_revenue))

## Display the rankings
display(ranked_products_df)

product,total_revenue,revenue_rank
Golden Gate Ginger,11595,1
Outback Oatmeal,11199,2
Austin Almond Biscotti,11148,3
Tokyo Tidbits,10986,4
Pearly Pies,10785,5
Orchard Oasis,10758,6



&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> | 
<a href="https://help.databricks.com/" target="blank">Support</a>
